# 04 — Calibration Experiments

Train and evaluate temperature scaling and Platt scaling on classifier outputs.

**Key metric:** Expected Calibration Error (ECE). Target: < 0.05 after calibration.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import sys
sys.path.insert(0, '../')

from app.calibration.temperature_scaling import TemperatureScaler
from app.calibration.platt_scaling import PlattScaler
from app.calibration.evaluator import compute_ece, plot_reliability_diagram

print('Calibration modules loaded.')

## 1. Load Validation Predictions

In [ ]:
# TODO: Replace with actual validation predictions from notebook 03
# Load the logits saved during training

# Simulated overconfident logits for demonstration
np.random.seed(42)
n_val = 500
n_classes = 9

# Simulate overconfident model
true_labels = np.random.randint(0, n_classes, n_val)
logits = np.random.randn(n_val, n_classes) * 3.0  # High variance = overconfident
# Make correct class have slightly higher logit
for i in range(n_val):
    logits[i, true_labels[i]] += 2.0

print(f'Validation set: {n_val} samples, {n_classes} classes')

## 2. Raw ECE (Before Calibration)

In [ ]:
from scipy.special import softmax

raw_probs = softmax(logits, axis=1)
raw_confidences = raw_probs.max(axis=1)
raw_predictions = raw_probs.argmax(axis=1)
raw_correct = (raw_predictions == true_labels).astype(float)

raw_ece = compute_ece(raw_confidences, raw_correct)
print(f"Raw ECE: {raw_ece['ece']:.4f}")
print(f"Raw accuracy: {raw_correct.mean():.4f}")

## 3. Temperature Scaling

In [ ]:
temp_scaler = TemperatureScaler()
temp_result = temp_scaler.fit(logits, true_labels)
print(f"Learned temperature: T = {temp_result['temperature']:.4f}")
print(f"NLL: {temp_result['nll_before']:.4f} → {temp_result['nll_after']:.4f}")

temp_probs = np.array([temp_scaler.calibrate(l) for l in logits])
temp_confidences = temp_probs.max(axis=1)
temp_ece = compute_ece(temp_confidences, raw_correct)
print(f"Temperature-scaled ECE: {temp_ece['ece']:.4f}")

## 4. Platt Scaling

In [ ]:
platt_scaler = PlattScaler()
platt_result = platt_scaler.fit(logits, true_labels)
print(f"Platt scaling accuracy: {platt_result['accuracy']:.4f}")

platt_probs = np.array([platt_scaler.calibrate(l) for l in logits])
platt_confidences = platt_probs.max(axis=1)
platt_ece = compute_ece(platt_confidences, raw_correct)
print(f"Platt-scaled ECE: {platt_ece['ece']:.4f}")

## 5. Compare All Three — Reliability Diagrams

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

for ax, (name, ece_result, confs) in zip(axes, [
    ('Raw', raw_ece, raw_confidences),
    ('Temperature Scaled', temp_ece, temp_confidences),
    ('Platt Scaled', platt_ece, platt_confidences),
]):
    bins = ece_result['bin_data']
    bin_confs = [b['avg_confidence'] for b in bins if b['count'] > 0]
    bin_accs = [b['avg_accuracy'] for b in bins if b['count'] > 0]
    
    ax.plot([0, 1], [0, 1], 'k--', alpha=0.5, label='Perfect')
    ax.bar(bin_confs, bin_accs, width=0.1, alpha=0.7, color='#7C3AED', edgecolor='#0d1117')
    ax.set_title(f'{name}\nECE = {ece_result["ece"]:.4f}', fontsize=12)
    ax.set_xlabel('Confidence')
    ax.set_ylabel('Accuracy')
    ax.set_xlim(0, 1)
    ax.set_ylim(0, 1)
    ax.legend()

plt.tight_layout()
plt.savefig('../data/calibration_comparison.png', dpi=150)
plt.show()
print('Saved calibration comparison to data/calibration_comparison.png')

## 6. Save Best Calibration Model

In [ ]:
# Choose best method
print(f'\n=== CALIBRATION RESULTS ===')
print(f'Raw ECE:              {raw_ece["ece"]:.4f}')
print(f'Temperature-scaled:   {temp_ece["ece"]:.4f}')
print(f'Platt-scaled:         {platt_ece["ece"]:.4f}')

best = 'temperature' if temp_ece['ece'] <= platt_ece['ece'] else 'platt'
print(f'\nBest method: {best}')

import os
os.makedirs('../ml_models/calibration', exist_ok=True)

if best == 'temperature':
    temp_scaler.save('../ml_models/calibration/temperature_scaler.pkl')
else:
    platt_scaler.save('../ml_models/calibration/platt_scaler.pkl')

print(f'Saved {best} scaler to ml_models/calibration/')
print('\nThese ECE numbers go directly into your research paper!')